In [1]:
import datetime as dt
import numpy as np
import pandas as pd 
import json
from shapely.geometry import Point, LineString, MultiLineString
import geopandas

In [2]:
colors_d = {
    'Ligne 1': '#FFCD00', 
    'Ligne 2': '#003CA6', 
    'Ligne 3': '#837902', 
    'Ligne 4': '#CF009E', 
    'Ligne 5': '#FF7E2E', 
    'Ligne 6': '#6ECA97',
    'Ligne 7': '#FA9ABA', 
    'Ligne 8': '#E19BDF',
    'Ligne 9': '#B6BD00',
    'Ligne 10': '#C9910D',
    'Ligne 11': '#704B1C',
    'Ligne A': '#2b5439',
    'Ligne 12': '#007852',
    'Ligne B': '#1891c4',
    'Ligne 13': '#6EC4E8',
    'Ligne 14': '#62259D',
    'Ligne 3 bis': '#6EC4E8',
    'Ligne 7 bis': '#6ECA97',
    'Couloirs': '#777779',
    'Ligne 2 Sud': '#ab1f07',
    'Ligne 14 (ancienne)': '#135e12',
    'Voie des Fêtes et voie navette': '#049fb0',
    # Grand Paris Express, official IDFM values.
    'Ligne 15': '#B90845',
    'Ligne 16': '#F3A4BA',
    'Ligne 17': '#D5C900',
    'Ligne 18': '#00A88F',
}

In [3]:
today = dt.datetime.combine(dt.date.today(), dt.time(0))

# Read data

In [4]:
file_path = 'data/raw_data/'
station_file_name = 'evolution_station.csv'
lignes_file_name = 'evolution_correspondances.csv'
futur_station_file_name = 'futur_station.csv'
futur_lignes_file_name = 'futur_correspondances.csv'

In [5]:
stations_evolution = pd.read_csv(file_path+station_file_name)
correspondances_evolution = pd.read_csv(file_path+lignes_file_name)

# Lines that have not opened live in their own files rather than behind a
# "start_date is in the future" test. A date test would quietly promote line 15
# to built the first time anyone rebuilt after its opening date, asserting as
# fact something nobody had checked; carrying the flag with the provenance means
# a projection stays a projection until someone moves the rows across.
stations_futur = pd.read_csv(file_path+futur_station_file_name)
correspondances_futur = pd.read_csv(file_path+futur_lignes_file_name)

stations_evolution['projet'] = False
correspondances_evolution['projet'] = False
stations_futur['projet'] = True
correspondances_futur['projet'] = True

stations_evolution = pd.concat([stations_evolution, stations_futur],
                               ignore_index=True)
correspondances_evolution = pd.concat([correspondances_evolution, correspondances_futur],
                                      ignore_index=True)

In [6]:
correspondances_evolution = correspondances_evolution.rename(
    columns={'Ouverture': 'start_date', 'Fermeture': 'end_date'}
)

In [7]:
stations_evolution['start_date'] = pd.to_datetime(stations_evolution['start_date'])
stations_evolution['end_date'] = pd.to_datetime(stations_evolution['end_date'])

In [8]:
correspondances_evolution['start_date'] = pd.to_datetime(correspondances_evolution['start_date'])
correspondances_evolution['end_date'] = pd.to_datetime(correspondances_evolution['end_date'])

# One sentinel closes everything that is still open, built and projected alike.
# It used to be the day the notebook ran, which worked while the data stopped at
# the present: now that the timeline reaches into 2028, that date would close the
# built network in the middle of the run — every station gaining a projected line
# would blink out on the sentinel date — and would close a projected line before
# it opened, dropping it from the output altogether. So the sentinel sits one day
# past the last opening in the data, or on the run date, whichever is later.
horizon = max(today,
              stations_evolution['start_date'].max() + dt.timedelta(days=1),
              correspondances_evolution['start_date'].max() + dt.timedelta(days=1))

stations_evolution['end_date'] = stations_evolution['end_date'].fillna(horizon)
correspondances_evolution['end_date'] = correspondances_evolution['end_date'].fillna(horizon)

# Lines to json

In [9]:
lines = correspondances_evolution['Ligne'].unique()
lines

array(['Ligne 1', 'Ligne 2', 'Ligne 2 Sud', 'Ligne 3', 'Ligne 4',
       'Ligne 5', 'Ligne 6', 'Ligne 7', 'Ligne 8', 'Ligne 9', 'Ligne 10',
       'Ligne 11', 'Ligne A', 'Ligne 12', 'Ligne B', 'Ligne 13',
       'Ligne 14', 'Ligne 3 bis', 'Ligne 7 bis', 'Ligne 14 (ancienne)',
       'Couloirs', 'Voie des Fêtes et voie navette'], dtype=object)

In [10]:
paths = []

# Grouped by (ligne, projet), not by ligne alone: the union below melts every
# segment of a snapshot into one geometry, so built and projected track have to
# be separated before it rather than after.
for line in lines:
    df_line = correspondances_evolution[(correspondances_evolution['Ligne']==line)]
    for projet in sorted(df_line['projet'].unique()):
        df = df_line[(df_line['projet']==projet)]
        key_dates = sorted(list(set(df['start_date'])|set(df['end_date'].dropna())))
        for i in range(len(key_dates)-1): 
            start_date = key_dates[i]
            end_date = key_dates[i+1]
            _df = df[(df['start_date']<=start_date)&(df['end_date']>start_date)]
            path = None
            for j in _df.index:
                de = _df.loc[j, 'De']
                vers = _df.loc[j, 'Vers']
                station_de = stations_evolution.loc[(stations_evolution['Nom de référence']==de)& 
                                                    (stations_evolution['start_date']<=start_date)&
                                                    (stations_evolution['end_date']>start_date)]
                station_vers = stations_evolution.loc[(stations_evolution['Nom de référence']==vers)& 
                                                      (stations_evolution['start_date']<=start_date)&
                                                      (stations_evolution['end_date']>start_date)]
                correspondance = LineString([Point([station_de['longitude'].values[0], 
                                                    station_de['latitude'].values[0]]), 
                                             Point([station_vers['longitude'].values[0], 
                                                    station_vers['latitude'].values[0]])
                                        ])
                if path == None: 
                    path = correspondance
                else: 
                    path = path.union(correspondance)

            paths.append({'ligne': line, 
                          'couleur': colors_d[line], 
                          'geometry': path, 
                          'start_date': start_date,
                          'end_date': end_date,
                          'projet': bool(projet)}
                         )

In [11]:
lignes_history = geopandas.GeoDataFrame(paths)
lignes_history['start_date'] = lignes_history['start_date'].map(lambda x: str(x.date()))
lignes_history['end_date'] = lignes_history['end_date'].map(lambda x: str(x.date()))

In [12]:
lignes_history.to_file("data/lignes_historiques.geojson", driver='GeoJSON')

# Stations to json

In [13]:
stations = stations_evolution['Nom de référence'].unique()

In [14]:
station_history = []

for station in stations: 
    station_evolution = stations_evolution[stations_evolution['Nom de référence'] == station]
    
    correspondance_evolution = correspondances_evolution[
        (correspondances_evolution['De']==station)
        ].groupby(['start_date', 'end_date'])['Ligne'].unique().reset_index()
    
    key_dates = sorted(set(station_evolution['start_date'])| 
                   set(station_evolution['end_date'])|
                   set(correspondance_evolution['start_date'])|
                   set(correspondance_evolution['end_date']))

    for i in range(len(key_dates)-1): 
            start_date = key_dates[i]
            end_date = key_dates[i+1]
            period_station_evolution = station_evolution.loc[
                (station_evolution['start_date']<=start_date)& 
                (station_evolution['end_date']>start_date)]
            if len(period_station_evolution):
                nom = period_station_evolution['Nom'].values[0]
                latitude = period_station_evolution['latitude'].values[0]
                longitude = period_station_evolution['longitude'].values[0]
                projet = bool(period_station_evolution['projet'].values[0])
                geometry = Point([longitude, latitude])
                lignes = correspondance_evolution.loc[(correspondance_evolution['start_date']<=start_date)& 
                                                      (correspondance_evolution['end_date']>start_date),
                                                      'Ligne'].values
                lignes = sorted(set([item for sublist in lignes for item in sublist]))
                if len(lignes)==1: 
                    couleur = colors_d[lignes[0]] 
                else:
                    couleur = '#D8D8B9'
                    
                station_step = {
                    'start_date': start_date,
                    'end_date': end_date, 
                    'nom': nom, 
                    'geometry': geometry, 
                    'lignes': lignes,
                    'couleur': couleur, 
                    'projet': projet, 
                    'nom de référence': station
                }

                if len(station_history)>1:
                    if (station_step['nom'] == station_history[-1]['nom'] and
                        station_step['lignes'] == station_history[-1]['lignes'] and 
                        station_step['geometry'] == station_history[-1]['geometry'] and
                        station_step['start_date'] == station_history[-1]['end_date']):
                        station_history[-1]['end_date'] = end_date 
                    else: 
                        station_history.append(station_step)
                else:
                    station_history.append(station_step)

In [15]:
station_history = geopandas.GeoDataFrame(station_history)

In [16]:
l = []
index_to_drop = []
for i in station_history.index:
    nom_i = station_history.at[i, 'nom']
    nom_de_ref_i = station_history.at[i, 'nom de référence']
    start_date_i = station_history.at[i, 'start_date']
    for j in station_history.index: 
        if j>i: 
            nom_j = station_history.at[j, 'nom']
            nom_de_ref_j = station_history.at[j, 'nom de référence']
            start_date_j = station_history.at[j, 'start_date']
            if (nom_j == nom_i and 
                nom_de_ref_j != nom_de_ref_i and
                start_date_j == start_date_i): 
                    l.append({'start_date': start_date_i, 'noms_de_ref': [nom_de_ref_j, nom_de_ref_i]})
                    index_to_drop.append(i)
                    index_to_drop.append(j)

In [17]:
events = []
for event in l: 
    start_date = event['start_date']
    station_i = event['noms_de_ref'][0]
    station_j = event['noms_de_ref'][1]
    events_i = station_history[(station_history['nom de référence']==station_i)&
                               (station_history['start_date']>=start_date)]
    events_j = station_history[(station_history['nom de référence']==station_j)&
                               (station_history['start_date']>=start_date)]
    key_dates = sorted(set(events_i['start_date'])| 
                       set(events_i['end_date'])|
                       set(events_j['start_date'])|
                       set(events_j['end_date']))
    for i in range(len(key_dates)-1): 
        start_date = key_dates[i]
        end_date = key_dates[i+1]
        info_i = events_i.loc[
                (events_i['start_date']<=start_date)& 
                (events_i['end_date']>start_date)].to_dict(orient='records')[0]
        info_j = events_j.loc[
                (events_j['start_date']<=start_date)& 
                (events_j['end_date']>start_date)].to_dict(orient='records')[0]
        lignes = info_i['lignes'] + info_j['lignes']
        lignes = sorted(set(lignes))
        
        for info in [info_i, info_j]: 
            info['lignes'] = lignes
            info['start_date'] = start_date
            info['end_date'] = end_date
            if len(lignes)==1:
                info['couleur'] = colors_d[lignes[0]]
            else: 
                '#D8D8B9'
            events.append(info)

In [18]:
station_history = pd.concat([station_history.drop(index_to_drop), 
                             pd.DataFrame(events)]).reset_index(drop=True)

In [19]:
station_history['lignes'] = station_history['lignes'].map(lambda x: str(x))
station_history['start_date'] = station_history['start_date'].map(lambda x: str(x.date()))
station_history['end_date'] = station_history['end_date'].map(lambda x: str(x.date()))

In [20]:
station_history[station_history['nom'] == 'Liège']

,couleur,end_date,geometry,lignes,nom,nom de référence,start_date
269,#1891c4,1931-03-27,POINT (2.32698 48.87963),['Ligne B'],Liège,Liège,1914-01-08
270,#6EC4E8,1939-09-03,POINT (2.32698 48.87963),['Ligne 13'],Liège,Liège,1931-03-27
271,#6EC4E8,2020-05-04,POINT (2.32698 48.87963),['Ligne 13'],Liège,Liège,1968-09-16


In [21]:
geopandas.GeoDataFrame(station_history).to_file("data/stations_historiques.geojson", driver='GeoJSON')